# 資料清洗（資料工程師）

讀 `input/raw.csv`（2500 列），寫出 `output/cleaned.csv`。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

Path("output").mkdir(exist_ok=True)
raw = pd.read_csv("input/raw.csv", dtype=str)
print("原始列數", len(raw))
raw.head()

In [ ]:
df = raw.copy()
log = {}

# 主鍵：沒有 ID 的列無法追溯，整列拒絕；重複 ID 保留第一筆
df["ID"] = pd.to_numeric(df["ID"], errors="coerce")
log["拒絕_無ID"] = int(df["ID"].isna().sum())
df = df[df["ID"].notna()]
log["拒絕_重複ID"] = int(df.duplicated("ID", keep="first").sum())
df = df.drop_duplicates("ID", keep="first")
df["ID"] = df["ID"].astype(int)

# AGE：負值、空白、unknown 一律 NULL，不猜（AGE 是受限欄位，不進模型）
age = pd.to_numeric(df["AGE"], errors="coerce")
log["AGE_設為NULL"] = int((age.isna() | (age < 1)).sum())
df["AGE"] = age.where(age >= 1)

# 婚姻狀態：只接受三個合法值，其他設 NULL
ms = df["MARITAL_STATUS"].str.strip().str.lower()
df["MARITAL_STATUS"] = ms.where(ms.isin(["single", "married", "divorced"]))
log["MARITAL_STATUS_設為NULL"] = int(df["MARITAL_STATUS"].isna().sum())

# PAY_0、BILL_AMT1：轉數字，帳單金額負值視為錯誤；缺值用中位數補並記錄筆數
df["PAY_0"] = pd.to_numeric(df["PAY_0"], errors="coerce")
bill = pd.to_numeric(df["BILL_AMT1"], errors="coerce")
df["BILL_AMT1"] = bill.where(bill >= 0)
for col in ["PAY_0", "BILL_AMT1"]:
    log[f"{col}_中位數補值"] = int(df[col].isna().sum())
    df[col] = df[col].fillna(df[col].median())

# 目標與洩漏欄：缺值補 0（洩漏欄保留在資料裡，但下游不可當特徵）
for col in ["default", "LEAK_FUTURE_DEFAULT"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# 時間：三種格式統一成 ISO-8601 UTC；解析不了的設 NULL
ts = pd.to_datetime(df["signup_at"], utc=True, errors="coerce", format="mixed")
log["signup_at_無法解析"] = int(ts.isna().sum())
df["signup_at"] = ts.dt.strftime("%Y-%m-%dT%H:%M:%SZ")

log["清洗後列數"] = len(df)
pd.Series(log)

In [ ]:
print("各欄缺值率")
print(df.isna().mean().round(4))
df.to_csv("output/cleaned.csv", index=False)
print("已寫出 output/cleaned.csv，", len(df), "列")

## 處理策略與影響筆數

| 欄位 | 問題 | 策略 | 影響筆數 |
|---|---|---|---|
| ID | 缺值、重複 | 缺值整列拒絕；重複保留第一筆 | 拒絕 1＋1 |
| AGE | 負值、空白、unknown | 一律 NULL，不猜 | 99 |
| MARITAL_STATUS | 非三個合法值 | NULL | 50 |
| PAY_0 | 無法轉數字 | 中位數補值 | 75 |
| BILL_AMT1 | 負值、無法轉數字 | 中位數補值 | 50 |
| signup_at | 三種格式混用 | 統一 ISO-8601 UTC；解析不了設 NULL | 1 |

清洗後 2498 列；除了 AGE（3.96%）、MARITAL_STATUS（2.00%）、signup_at（0.04%）以外各欄缺值率都是 0。重跑方式：直接執行本 notebook，沒有隨機性，輸入相同就得到同一份 cleaned.csv。

## 拒絕列與原因

共拒絕 2 列：1 列沒有 ID（主鍵缺失，無法追溯到申請案，補值等於捏造身分）；1 列 ID 重複（保留第一筆，兩筆內容相同，判斷是系統重複上傳）。沒有任何列是靜默刪除的，數字都在上面程式輸出裡。

## 給下游的限制與假設

1. AGE、MARITAL_STATUS 是法遵受限欄位，我保留 NULL 沒有補值，**下游不可當特徵**。
2. LEAK_FUTURE_DEFAULT 是目標洩漏欄，留在資料裡只供稽核，**不可進模型**。
3. PAY_0 有 75 筆、BILL_AMT1 有 50 筆是中位數補的，DS 做特徵重要性時要知道這 3% 的值不是真的觀測值。
4. PAY_0 是本主題最關鍵的欄位，補值會把少數人拉到「準時」附近，可能讓高風險群的違約率略被低估。